# 诗歌生成

# 数据处理

In [ ]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
from tqdm.auto import tqdm

# GPU setup: enable memory growth and show available devices.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('Using GPU(s):', gpus)
    except RuntimeError as e:
        print('GPU config skipped:', e)
else:
    print('No GPU found, will run on CPU.')

start_token = 'bos'
end_token = 'eos'

# Smaller limits reduce sequence padding and per-step memory usage.
MAX_SEQ_LEN = 120
BATCH_SIZE = 8

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > MAX_SEQ_LEN:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('./poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(BATCH_SIZE, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

Using GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# 模型代码， 完成建模代码

In [ ]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 32)
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(64)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)


    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        emb = self.embed_layer(inp_ids) #shape(b_sz, seq_len, emb_sz)
        rnn_out = self.rnn_layer(emb) #shape(b_sz, seq_len, h_sz)
        logits = self.dense(rnn_out) #shape(b_sz, seq_len, v_sz)
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
    
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, [state]) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state[0]

## 一个计算sequence loss的辅助函数，只需了解用途。

In [14]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [15]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)


def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    cardinality = tf.data.experimental.cardinality(ds).numpy()
    steps_per_epoch = int(cardinality) if cardinality >= 0 else None

    pbar = tqdm(ds, total=steps_per_epoch, desc=f'Epoch {epoch}', leave=True)
    for step, (x, y, seqlen) in enumerate(pbar):
        loss = train_one_step(model, optimizer, x, y, seqlen)
        pbar.set_postfix(loss=float(loss.numpy()))

    return loss

# 训练优化过程

In [ ]:
train_ds, word2id, id2word = poem_dataset()

# Place model/optimizer creation and training on GPU when available.
device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print('Training device:', device)

with tf.device(device):
    optimizer = optimizers.SGD(learning_rate=0.01)
    model = myRNNModel(word2id)
    for epoch in range(10):
        loss = train(epoch, model, optimizer, train_ds)

Training device: /GPU:0


Epoch 0: 8it [00:11,  1.14s/it, loss=8.78]

Epoch 0: 13it [00:17,  1.13s/it, loss=8.64]

Epoch 0: 731it [14:16,  1.17s/it, loss=6.46]


ResourceExhaustedError: {{function_node __wrapped__ResourceScatterAdd_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating a buffer of 2906488576 bytes [Op:ResourceScatterAdd]

# 生成过程

In [ ]:
def gen_sentence():
    state = tf.zeros(shape=(1, 64), dtype=tf.float32)
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        collect.append(cur_token.numpy()[0])
    return [id2word[t] for t in collect]
print(''.join(gen_sentence()))

NameError: name 'tf' is not defined

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('Built with CUDA:', tf.test.is_built_with_cuda())
print('Physical GPUs:', tf.config.list_physical_devices('GPU'))
print('Logical GPUs:', tf.config.list_logical_devices('GPU'))

TF version: 2.20.0
Built with CUDA: False
Physical GPUs: []
Logical GPUs: []
